# Análise de Turnover: Testes de Hipótese e Análise Exploratória

**Pergunta de negócio:** o que está associado ao desligamento de colaboradores e em quais áreas o problema se concentra?

**Base:** tabela `rh_dados_limpos` (gerada no `01_elt.ipynb`), com uma linha por colaborador (650 registros).

**Definição de turnover:** proporção de colaboradores com `status_atual = 'Desligado'` na base. Como não há data de desligamento, **não é uma taxa anual**, e sim uma fotografia do quadro.

In [1]:
import pandas as pd
import duckdb
from scipy import stats

con = duckdb.connect('../data/rh_dados.duckdb')

df_analise = con.execute("SELECT * FROM rh_dados_limpos").df()

In [2]:
ativos = df_analise[df_analise['desligado'] == 0]
desligados = df_analise[df_analise['desligado'] == 1]

## Hipóteses

|  | Hipótese | Variáveis | Método |
|---|---|---|---|
| H1 | Sobrecarga de trabalho está associada ao desligamento | `horas_extras_mes`, `processos_ativos` | Comparação de médias |
| H2 | Insatisfação está associada ao desligamento | `score_satisfacao` | Comparação de médias |
| H3 | Há viés de gênero, modelo de trabalho ou localização | `genero`, `home_office`, `estado` | Qui-quadrado |
| H4 | A ausência de promoção está associada ao desligamento | `promocao` | Qui-quadrado |

Nível de significância: **α = 0,05**.

## Sobrecarga e satisfação (H1 e H2)


In [3]:
print("\n SOBRECARGA DE TRABALHO E BURNOUT (Análise Descritiva) \n")

media_horas_ativo = ativos['horas_extras_mes'].mean()
media_horas_deslig = desligados['horas_extras_mes'].mean()
media_proc_ativo = ativos['processos_ativos'].mean()
media_proc_deslig = desligados['processos_ativos'].mean()

print(f" Horas extras por mês = Ativos: {media_horas_ativo:.1f}h | Desligados: {media_horas_deslig:.1f}h")
print(f" Processos Ativos = Ativos: {media_proc_ativo:.1f}  | Desligados: {media_proc_deslig:.1f}")


 SOBRECARGA DE TRABALHO E BURNOUT (Análise Descritiva) 

 Horas extras por mês = Ativos: 18.9h | Desligados: 26.4h
 Processos Ativos = Ativos: 30.5  | Desligados: 36.5


In [4]:
print("\n CLIMA ORGANIZACIONAL E INSATISFAÇÃO (Análise Descritiva) \n")

sat_ativo = ativos['score_satisfacao'].mean()
sat_deslig = desligados['score_satisfacao'].mean()

print(f" Score de Satisfação = Ativos: {sat_ativo:.2f} / 5.0 | Desligados: {sat_deslig:.2f} / 5.0")


 CLIMA ORGANIZACIONAL E INSATISFAÇÃO (Análise Descritiva) 

 Score de Satisfação = Ativos: 3.62 / 5.0 | Desligados: 3.04 / 5.0


Desligados registram **26.4 h** extras/mês contra **18.9**h dos ativos, e **36.5** processos ativos contra **30.5**. A satisfação média é de **3.04** entre desligados e **3.62** entre ativos (escala de 1 a 5). Esta é uma **comparação descritiva**.

**Conclusão Descritiva:** Colaboradores desligados apresentavam, em média, um volume notavelmente maior de sobrecarga operacional e o nível médio de insatisfação é visivelmente maior no grupo de profissionais desligados.

## Testes Qui-Quadrado (H3 e H4)

O **Teste Qui-Quadrado** verifica se duas variáveis categóricas estão associadas. Um p-valor < 0,05 indica evidência de associação, mas não mede a intensidade nem prova causalidade. Um p-valor ≥ 0,05 indica que **não há evidência de associação**, o que não equivale a provar que ela não existe, mas pode diminuir a prioridade de análise.

In [5]:
%run -i ../src/testes_estatisticos.py


 INVESTIGAÇÃO DE VIESES (GÊNERO, HOME OFFICE E LOCALIZAÇÃO) 

 Gênero 

 Taxa de Turnover (Não-binário) : 30.4%
 Taxa de Turnover (Masculino) : 25.4%
 Taxa de Turnover (Feminino) : 26.0%
 Teste Qui-Quadrado (Gênero): p-valor = 0.4461410262
 O gênero não afeta o desligamento (Hipótese de viés descartada)

 Modelo de Trabalho 

 Taxa de Turnover (Home Office) : 27.9%
 Taxa de Turnover (Presencial)  : 26.8%
 Teste Qui-Quadrado (Home Office): p-valor = 0.8103327482
 A modalidade de trabalho não afeta o desligamento (Hipótese descartada)

 Localização 

 Taxa de Turnover (BA) : 26.5%
 Taxa de Turnover (MG) : 23.7%
 Taxa de Turnover (RJ) : 29.5%
 Taxa de Turnover (SP) : 28.9%
 Teste Qui-Quadrado (Estado): p-valor = 0.6367120196
 A localização geográfica não impacta o desligamento (Hipótese descartada)


 ESTAGNAÇÃO DE CARREIRA E FALTA DE PROMOÇÃO (Teste de Hipótese) 

 Taxa de Turnover (Nunca promovidos) : 36.6%
 Taxa de Turnover (Já promovidos)    : 8.3%
 Teste Qui-Quadrado (Chi2): p-valor

| Variável | Turnover por grupo | p-valor | Leitura |
|---|---|---|---|
| Gênero | Não-binário 30,4% · Masculino 25,4% · Feminino 26,0% | 0,446 | Sem evidência de associação |
| Modelo de trabalho | Home office 27,9% · Presencial 26,8% | 0,810 | Sem evidência de associação |
| Localização | BA 26,5% · MG 23,7% · RJ 29,5% · SP 28,9% | 0,637 | Sem evidência de associação |
| Promoção | Nunca promovidos 36,6% · Já promovidos 8,3% | < 0,001 | **Associação forte** |


## Análise exploratória por departamento e senioridade

Os problemas de sobrecarga e de falta de promoção são gerais ou concentrados em áreas específicas? As tabelas abaixo mostram o recorte por departamento e nível. Percentuais de subgrupos pequenos são instáveis, por isso a presença do total de pessoas é importante.

### Mapeamento Departamental

In [6]:
query_departamental = """
SELECT 
    nome_departamento AS Departamento, 
    COUNT(id_colaborador) AS Total_Colaboradores,
    SUM(desligado) AS Total_Desligados,
    ROUND(AVG(desligado) * 100, 1) AS Taxa_Turnover,
    ROUND(AVG(horas_extras_mes), 1) AS Horas_Extras_Medias,
    ROUND(AVG(score_satisfacao), 2) AS Satisfacao_Media,
    ROUND(AVG(promocao) * 100, 1) AS Promovidos
FROM rh_dados_limpos f
    GROUP BY nome_departamento
    ORDER BY Taxa_Turnover DESC;
"""

df_departamento = con.execute(query_departamental).df()
df_departamento


,Departamento,Total_Colaboradores,Total_Desligados,Taxa_Turnover,Horas_Extras_Medias,Satisfacao_Media,Promovidos
0,Trabalhista,170,74.0,43.5,46.3,1.67,30.6
1,Tributário,182,50.0,27.5,12.9,4.18,32.4
2,Civil,151,28.0,18.5,12.0,4.10,35.1
3,Corporativo,147,25.0,17.0,10.7,4.07,35.4


### Insights de mapeamento departamental


* **Setor Trabalhista** : Responde pela maior taxa de turnover da empresa (43.5%), impulsionado por uma média de 46.3 horas extras/mês por colaborador. Registrando a menor nota de satisfação do escritório (1.67 / 5.0).
* **Setor Tributário** : Apresenta a segunda maior taxa de evasão (27.5%), contudo mantém altos índices de satisfação (4.18) e volume moderado de horas extras (12.9h). Isso sugere que a perda de talentos no Tributário não é motivada por sobrecarga, mas por outros motivos como defasagem na remuneração.
* **Civil e Corporativo** : Apresentam turnover controlado (17% a 18%), baixas horas extras (11h) e score de satisfação elevado (> 4.0).

### Aprofundando a análise dos departamentos

Entendendo melhor a situação do setor trabalhista e do setor tributário

#### Mapeamento do Setor Trabalhista

In [7]:
query_trabalhista = """
SELECT 
    nivel AS Senioridade,
    COUNT(id_colaborador) AS Total,
    SUM(desligado) AS Desligados,
    ROUND(AVG(desligado) * 100, 1) AS Turnover,
    ROUND(AVG(horas_extras_mes), 1) AS Horas_Extras,
    ROUND(AVG(score_satisfacao), 2) AS Satisfacao
FROM rh_dados_limpos
WHERE nome_departamento = 'Trabalhista'
GROUP BY nivel;
"""

df_trabalhista = con.execute(query_trabalhista).df()
df_trabalhista

,Senioridade,Total,Desligados,Turnover,Horas_Extras,Satisfacao
0,Júnior,81,43.0,53.1,45.7,1.64
1,Pleno,60,29.0,48.3,46.9,1.68
2,Sênior,23,2.0,8.7,46.7,1.73
3,Sócio,6,0.0,0.0,46.5,1.70


#### Setor Trabalhista
**O que os dados mostram:**
* Maior turnover da empresa (43,5%), com média de 46,3h extras/mês e a menor satisfação (1,67/5,0).
* A sobrecarga e a insatisfação são homogêneas entre os níveis, mas a saída se concentra na base: Júnior 53,1% e Pleno 48,3%, contra Sênior 8,7% e Sócio 0,0%.

O modelo operacional do Trabalhista sobrecarrega a base de execução, tornando o setor um ponto de atrito para profissionais em início e meio de carreira. Sêniores permanecem apesar do desgaste.

#### Mapeamento do Setor Tributário

In [8]:
query_tributario = """
SELECT 
    nivel AS Senioridade,
    COUNT(id_colaborador) AS Total,
    SUM(desligado) AS Desligados,
    ROUND(AVG(desligado) * 100, 1) AS Turnover,
    ROUND(AVG(horas_extras_mes), 1) AS Horas_Extras,
    ROUND(AVG(score_satisfacao), 2) AS Satisfacao
FROM rh_dados_limpos
WHERE nome_departamento = 'Tributário'
GROUP BY nivel;
"""

df_tributario = con.execute(query_tributario).df()
df_tributario

,Senioridade,Total,Desligados,Turnover,Horas_Extras,Satisfacao
0,Pleno,73,39.0,53.4,13.1,4.20
1,Sênior,31,11.0,35.5,12.1,4.19
2,Sócio,9,0.0,0.0,13.6,4.08
3,Júnior,69,0.0,0.0,12.9,4.16


In [9]:
query_tributario = """
SELECT 
    status_atual,
    COUNT(id_colaborador) AS Total,
    ROUND(AVG(remuneracao_total), 2) AS Remuneracao_Media,
    ROUND(AVG(bonus) * 100, 1) AS Pct_Recebeu_Bonus,
    ROUND(AVG(promocao) * 100, 1) AS Pct_Promovidos
FROM rh_dados_limpos
WHERE nome_departamento = 'Tributário'
GROUP BY status_atual;
"""

df_tributario = con.execute(query_tributario).df()
df_tributario

,status_atual,Total,Remuneracao_Media,Pct_Recebeu_Bonus,Pct_Promovidos
0,Desligado,50,9502.46,100.0,22.0
1,Ativo,132,8895.20,96.2,36.4


### Setor Tributário
**O que os dados mostram:**
* Turnover de 27,5% com satisfação alta (4,18) e poucas horas extras (12,9h): sobrecarga e clima não explicam as saídas.
* Desligados têm remuneração média maior (R$ 9.502,46 vs R$ 8.895,20).
* Desligados foram menos promovidos (22,0% vs 36,4% dos ativos).

A perda no Tributário pode ocorrer por falta de perspectiva de progressão de carreira, não por sobrecarga ou insatisfação financeira.

## Cenário de Senioridade Departamental

In [10]:
query_senioridade_departamento = """
SELECT 
    nome_departamento AS Departamento,
    nivel AS Senioridade,
    ROUND(AVG(remuneracao_total), 2) AS Remuneracao_Total,
    ROUND(AVG(horas_extras_mes), 1) AS Horas_Extras,
    ROUND(AVG(processos_ativos), 0) AS Processos_Ativos,
    ROUND(AVG(desligado) * 100, 1) AS Turnover,
    ROUND(AVG(promocao) * 100, 1) AS Promovidos
FROM rh_dados_limpos
WHERE nivel IN ('Júnior', 'Pleno', 'Sênior')
GROUP BY nome_departamento, nivel
ORDER BY Senioridade ASC;
"""

df_senioridade = con.execute(query_senioridade_departamento).df()
df_senioridade

,Departamento,Senioridade,Remuneracao_Total,Horas_Extras,Processos_Ativos,Turnover,Promovidos
0,Corporativo,Júnior,4406.25,10.9,25.0,0.0,0.0
1,Civil,Júnior,4429.51,12.2,24.0,0.0,0.0
2,Trabalhista,Júnior,4535.34,45.7,57.0,53.1,0.0
3,Tributário,Júnior,4450.58,12.9,24.0,0.0,0.0
4,Trabalhista,Pleno,7985.66,46.9,55.0,48.3,45.0
5,Corporativo,Pleno,7824.22,10.8,23.0,42.4,45.8
6,Civil,Pleno,8321.84,12.3,23.0,42.6,42.6
7,Tributário,Pleno,8110.38,13.1,23.0,53.4,30.1
8,Trabalhista,Sênior,14929.37,46.7,57.0,8.7,82.6
9,Corporativo,Sênior,15424.86,11.0,23.0,0.0,90.0


#### Cenário Geral de Senioridade Departamental

Comparar departamentos **dentro do mesmo nível** isola melhor cada fator, e mostra que o turnover tem causas diferentes em cada etapa da carreira.

**Júnior:** A remuneração é praticamente igual em todas as áreas (R$ 4.406 a R$ 4.535) e nenhum Júnior foi promovido. Só o Trabalhista se destaca em carga (45,7h extras e 57 processos, contra 11 a 13h e 24 a 25 processos nos demais), e é o único com turnover: 53,1% contra 0,0%. É a comparação mais limpa da base, a carga de trabalho é o diferencial.

**Pleno:** Varia de 42,4% a 53,4% em todos os departamentos, inclusive onde as horas extras são baixas (~11 a 13h), então a sobrecarga não explica este nível. O Tributário reúne o maior turnover (53,4%) e a menor taxa de promoção (30,1% contra 42,6% a 45,8%), o que é coerente com a de que o Pleno, já dominando a operação, seja o perfil mais disputado pelo mercado e o mais sensível à progressão.

**Sênior:** A remuneração é semelhante (R$ 14.448 a R$ 15.425). O Trabalhista, com a maior carga, tem só 8,7% de turnover, enquanto o Tributário, com carga baixa, tem 35,5%. A remuneração interna também não explica: o Tributário paga mais que o Civil e perde mais gente. o Turnover depende do departamento, e não da carga ou do salário

**No Trabalhista, a carga é a mesma em todos os níveis**, mas o turnover cai de 53,1% (Júnior) para 48,3% (Pleno) e 8,7% (Sênior). A sobrecarga parece necessária, mas não suficiente. A hipótese a investigar é que remuneração e progressão moderem o efeito.

In [11]:
query_senioridade_departamento = """
SELECT 
    nome_departamento AS Departamento,
    COUNT(id_colaborador) AS Total_Seniores,
    SUM(desligado) AS Seniores_Desligados,
    ROUND(AVG(desligado) * 100, 1) AS Turnover_Senior,
    ROUND(AVG(CASE WHEN status_atual = 'Ativo' THEN remuneracao_total END), 2) AS Salario_Ativo,
    ROUND(AVG(CASE WHEN status_atual = 'Desligado' THEN remuneracao_total END), 2) AS Salario_Desligado,
    ROUND(AVG(horas_extras_mes), 1) AS Horas_Extras_Medias,
    ROUND(AVG(score_satisfacao), 2) AS Satisfacao_Media
FROM rh_dados_limpos
WHERE nivel = 'Sênior'
GROUP BY nome_departamento
ORDER BY Seniores_Desligados DESC;
"""

df_senioridade = con.execute(query_senioridade_departamento).df()
df_senioridade

,Departamento,Total_Seniores,Seniores_Desligados,Turnover_Senior,Salario_Ativo,Salario_Desligado,Horas_Extras_Medias,Satisfacao_Media
0,Tributário,31,11.0,35.5,15267.56,14737.73,12.1,4.19
1,Civil,21,5.0,23.8,14490.42,14312.07,10.6,4.14
2,Trabalhista,23,2.0,8.7,15050.95,13652.75,46.7,1.73
3,Corporativo,20,0.0,0.0,15424.86,NaN,11.0,4.13


#### Cenário de Turnover de Sêniores

O turnover varia muito por departamento: Tributário 35,5% (11 de 31), Civil 23,8% (5 de 21), Trabalhista 8,7% (2 de 23) e Corporativo 0,0% (0 de 20). Como os grupos são pequenos, cada saída pesa vários pontos percentuais, e as comparações abaixo devem ser feitas com cautela.

* **Tributário, Civil e Corporativo.** Os três têm carga de trabalho (10,6h a 12,1h extras), satisfação (4,13 a 4,19), remuneração dos ativos (R$ 14,5 mil a R$ 15,4 mil) e taxa de promoção (~90%) praticamente iguais, mas turnover de 35,5%, 23,8% e 0,0%. As variáveis disponíveis não explicam.
* **Tributário (35,5%):** o maior turnover sênior ocorre com satisfação alta (4,19) e baixa carga (12,1h), então sobrecarga e clima não explicam as saídas. Os desligados ganhavam em média 3,5% menos que os ativos (R$ 14.737,73 vs R$ 15.267,56), diferença pequena que, com 11 desligados, não permite concluir por defasagem salarial. Hipóteses possíveis são atratividade do mercado externo e fatores internos não medidos.
* **Trabalhista (8,7%, 2 desligados):** convivem a maior carga (46,7h) e a menor satisfação (1,73) com baixo turnover sênior. Com apenas 2 casos, esse valor não se distingue estatisticamente de 0% e a causa dessa retenção permanece em aberto.
* **Corporativo (0,0%, n = 20):** nenhum desligamento com a maior remuneração entre os ativos (R$ 15.424,86).
* **Padrão salarial:** em todos os departamentos com desligamentos, os desligados ganhavam menos que os ativos do mesmo nível. É um sinal fraco, porque os grupos são pequenos, mas consistente na direção.

### Limitação de Dados

* **Sem data de desligamento ou tempo de casa:** impede curvas de retenção e dificulta interpretar a associação com promoção.
* **Sem motivo do desligamento:** a base não separa saída voluntária de demissão pela empresa.
* **Associações, não causas:** as explicações propostas são hipóteses.
* **Cuidado com a variável "Promovidos":** ela cresce com o nível (0% nos Júniores, 30 a 46% nos Plenos, 83 a 90% nos Sêniores). A associação global entre promoção e turnover (36,6% contra 8,3%) mistura, portanto, o efeito da promoção com o do nível. 
